# Part 3 - Adversarial Attacks: Breaking the Classifier

This notebook implements two attacks from scratch:
1. Character-level evasion attack (zero-width spaces + homoglyphs + duplication)
2. Label-flipping poisoning attack (5% label flips + retraining)

Deliverables produced in outputs:
- Attack 1 ASR table
- Attack 2 before/after metric comparison (Accuracy, F1, FNR)

In [ ]:
# Uncomment in a fresh environment.
# !pip install -q transformers datasets accelerate torch scikit-learn pandas matplotlib seaborn

In [1]:
import os
import re
import zipfile
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Libraries imported.")
print(f"CUDA available: {torch.cuda.is_available()}")

Libraries imported.
CUDA available: True


In [2]:
import zipfile
with zipfile.ZipFile("distilbert_toxicity_checkpoint_part_1.zip", 'r') as zip_ref:
    zip_ref.extractall()

In [3]:
# Step 1: Paths and model loading (clean Part 1 model).
EVAL_PATH = "eval_subset_20k.csv"
TRAIN_PATH = "train_subset_100k.csv"
BASE_MODEL = "distilbert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE_INFER = 64
OPERATING_THRESHOLD = 0.5

# Use only this folder for the clean Part 1 model.
MODEL_DIR = "distilbert_toxicity_checkpoint_part_1"

if not os.path.isdir(MODEL_DIR):
    raise FileNotFoundError(
        f"Model directory not found: {MODEL_DIR}. "
        "Make sure the Part 1 checkpoint folder exists in the workspace."
    )

print(f"Using clean model directory: {MODEL_DIR}")

Using clean model directory: distilbert_toxicity_checkpoint_part_1


In [4]:
# Step 2: Load clean eval/train subsets and initialize tokenizer/model.
eval_df = pd.read_csv(EVAL_PATH)
train_df = pd.read_csv(TRAIN_PATH)

if "label" not in eval_df.columns:
    eval_df["label"] = (eval_df["toxic"] >= 0.5).astype(int)
if "label" not in train_df.columns:
    train_df["label"] = (train_df["toxic"] >= 0.5).astype(int)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
clean_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

print(f"Eval rows: {len(eval_df):,}")
print(f"Train rows: {len(train_df):,}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Eval rows: 20,000
Train rows: 100,000


In [5]:
# Shared inference + metrics helpers.
def predict_probs(model, tokenizer, texts, batch_size=64, max_length=128):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    probs_all = []
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
        probs_all.append(probs)

    return np.concatenate(probs_all)

def compute_core_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="binary")
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
    return {"Accuracy": acc, "F1": f1, "FNR": fnr, "TN": tn, "FP": fp, "FN": fn, "TP": tp}

## Attack 1 - Character-Level Evasion

In [7]:
# Step 3: Build perturb(text) with all three required transformations.
ZERO_WIDTH = "\u200b"
TOXIC_HINT_WORDS = {
    "hate", "stupid", "idiot", "dumb", "moron", "trash", "ugly", "kill",
    "racist", "nazi", "loser", "bitch", "fool", "suck", "damn", "toxic"
}
HOMOGLYPHS = {
    "a": "ӓ",  # Cyrillic a
    "e": "ě",  # Cyrillic e
    "o": "0",  # Cyrillic o
    "A": "Ǎ",
    "E": "Ě",
    "O": "𐓃",
}

def insert_zero_width(word, rng):
    if len(word) < 4:
        return word
    out = []
    i = 0
    while i < len(word):
        step = 2 if rng.random() < 0.5 else 3
        out.append(word[i:i+step])
        i += step
    return ZERO_WIDTH.join(out)

def homoglyph_substitute(word, rng, p=0.35):
    chars = []
    for ch in word:
        if ch in HOMOGLYPHS and rng.random() < p:
            chars.append(HOMOGLYPHS[ch])
        else:
            chars.append(ch)
    return "".join(chars)

def random_duplicate(word, rng, p=0.20):
    chars = []
    for ch in word:
        chars.append(ch)
        if ch.isalpha() and rng.random() < p:
            chars.append(ch)
    return "".join(chars)

def perturb(text, seed=42):
    rng = random.Random(seed + (hash(text) % 10_000_000))

    tokens = re.findall(r"\w+|\W+", text, flags=re.UNICODE)
    transformed = []

    for tok in tokens:
        if not re.match(r"^\w+$", tok, flags=re.UNICODE):
            transformed.append(tok)
            continue

        w = tok
        w_lower = w.lower()

        if any(hint in w_lower for hint in TOXIC_HINT_WORDS):
            w = insert_zero_width(w, rng)

        w = homoglyph_substitute(w, rng, p=0.35)
        w = random_duplicate(w, rng, p=0.20)

        transformed.append(w)

    return "".join(transformed)

print(perturb("I hate this stupid toxic comment"))

I hhӓӓ​te tthis sstu​piid tto​xxiic coomměnt


In [8]:
# Step 4: Select 500 toxic comments predicted positive with confidence >= 0.7.
eval_texts = eval_df["comment_text"].fillna("").astype(str).tolist()
clean_probs_eval = predict_probs(clean_model, tokenizer, eval_texts, BATCH_SIZE_INFER, MAX_LENGTH)
eval_df["clean_prob"] = clean_probs_eval
eval_df["clean_pred"] = (eval_df["clean_prob"] >= OPERATING_THRESHOLD).astype(int)

attack_pool = eval_df[(eval_df["clean_pred"] == 1) & (eval_df["clean_prob"] >= 0.7)].copy()
print(f"Eligible pool size for attack: {len(attack_pool):,}")

sample_n = min(500, len(attack_pool))
attack_sample = attack_pool.sample(n=sample_n, random_state=SEED).copy()
print(f"Attack sample size used: {len(attack_sample):,}")

Eligible pool size for attack: 1,277
Attack sample size used: 500


In [9]:
# Step 5: Apply perturbation and compute Attack Success Rate (ASR).
attack_sample["perturbed_text"] = attack_sample["comment_text"].fillna("").astype(str).apply(perturb)

perturbed_probs = predict_probs(
    clean_model,
    tokenizer,
    attack_sample["perturbed_text"].tolist(),
    BATCH_SIZE_INFER,
    MAX_LENGTH,
)

attack_sample["perturbed_prob"] = perturbed_probs
attack_sample["perturbed_pred"] = (attack_sample["perturbed_prob"] >= OPERATING_THRESHOLD).astype(int)

flipped_to_non_toxic = (attack_sample["perturbed_pred"] == 0).sum()
asr = flipped_to_non_toxic / len(attack_sample) if len(attack_sample) > 0 else np.nan

asr_table = pd.DataFrame([{
    "sample_size": len(attack_sample),
    "attack_success_count": int(flipped_to_non_toxic),
    "attack_success_rate": asr,
    "avg_conf_before": attack_sample["clean_prob"].mean(),
    "avg_conf_after": attack_sample["perturbed_prob"].mean(),
}])

print("Attack 1 (Character-level evasion) ASR table")
display(asr_table)

Attack 1 (Character-level evasion) ASR table


,sample_size,attack_success_count,attack_success_rate,avg_conf_before,avg_conf_after
0,500,416,0.832,0.9259,0.214142


## Attack 2 - Label-Flipping Poisoning

In [10]:
# Step 6: Create poisoned train set with 5% random label flips.
poisoned_train = train_df.copy()
flip_frac = 0.05
flip_n = int(len(poisoned_train) * flip_frac)
flip_idx = poisoned_train.sample(n=flip_n, random_state=SEED).index

poisoned_train.loc[flip_idx, "label"] = 1 - poisoned_train.loc[flip_idx, "label"]

print(f"Total train rows: {len(poisoned_train):,}")
print(f"Flipped rows (5%): {flip_n:,}")

Total train rows: 100,000
Flipped rows (5%): 5,000


In [11]:
# Step 7: Retrain DistilBERT on poisoned data with Part 1 hyperparameters.
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
BATCH_SIZE_TRAIN = 16
POISONED_OUT_DIR = "distilbert_poisoned_checkpoint_part3"

train_hf = Dataset.from_pandas(poisoned_train[["comment_text", "label"]].reset_index(drop=True))
eval_hf = Dataset.from_pandas(eval_df[["comment_text", "label"]].reset_index(drop=True))

def tokenize_batch(batch):
    return tokenizer(batch["comment_text"], truncation=True, max_length=MAX_LENGTH)

train_tok = train_hf.map(tokenize_batch, batched=True, remove_columns=["comment_text"])
eval_tok = eval_hf.map(tokenize_batch, batched=True, remove_columns=["comment_text"])

train_tok = train_tok.rename_column("label", "labels")
eval_tok = eval_tok.rename_column("label", "labels")

train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
poisoned_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)

args_common = dict(
    output_dir=POISONED_OUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE_TRAIN,
    per_device_eval_batch_size=BATCH_SIZE_TRAIN,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to="none",
)

try:
    poisoned_args = TrainingArguments(evaluation_strategy="epoch", **args_common)
except TypeError:
    poisoned_args = TrainingArguments(eval_strategy="epoch", **args_common)

try:
    poisoned_trainer = Trainer(
        model=poisoned_model,
        args=poisoned_args,
        train_dataset=train_tok,
        eval_dataset=eval_tok,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
except TypeError:
    poisoned_trainer = Trainer(
        model=poisoned_model,
        args=poisoned_args,
        train_dataset=train_tok,
        eval_dataset=eval_tok,
        data_collator=data_collator,
    )

poisoned_train_result = poisoned_trainer.train()
print(poisoned_train_result)

poisoned_trainer.save_model(POISONED_OUT_DIR)
print(f"Saved poisoned model to: {POISONED_OUT_DIR}")

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=18750, training_loss=0.26363587544759115, metrics={'train_runtime': 1343.5612, 'train_samples_per_second': 223.287, 'train_steps_per_second': 13.955, 'total_flos': 9848586804727296.0, 'train_loss': 0.26363587544759115, 'epoch': 3.0})


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved poisoned model to: distilbert_poisoned_checkpoint_part3


In [12]:
# Step 8: Evaluate clean vs poisoned model on clean eval subset.
eval_texts = eval_df["comment_text"].fillna("").astype(str).tolist()
y_true = eval_df["label"].astype(int).values

clean_probs = predict_probs(clean_model, tokenizer, eval_texts, BATCH_SIZE_INFER, MAX_LENGTH)
clean_pred = (clean_probs >= OPERATING_THRESHOLD).astype(int)
clean_metrics = compute_core_metrics(y_true, clean_pred)

poisoned_probs = predict_probs(poisoned_model, tokenizer, eval_texts, BATCH_SIZE_INFER, MAX_LENGTH)
poisoned_pred = (poisoned_probs >= OPERATING_THRESHOLD).astype(int)
poisoned_metrics = compute_core_metrics(y_true, poisoned_pred)

comparison = pd.DataFrame([
    {"Model": "Clean baseline", "Accuracy": clean_metrics["Accuracy"], "F1": clean_metrics["F1"], "FNR": clean_metrics["FNR"]},
    {"Model": "Poisoned (5% flip)", "Accuracy": poisoned_metrics["Accuracy"], "F1": poisoned_metrics["F1"], "FNR": poisoned_metrics["FNR"]},
])

delta_row = pd.DataFrame([{
    "Model": "Change (poisoned - clean)",
    "Accuracy": poisoned_metrics["Accuracy"] - clean_metrics["Accuracy"],
    "F1": poisoned_metrics["F1"] - clean_metrics["F1"],
    "FNR": poisoned_metrics["FNR"] - clean_metrics["FNR"],
}])

print("Attack 2 before/after metric comparison")
display(pd.concat([comparison, delta_row], ignore_index=True))

Attack 2 before/after metric comparison


,Model,Accuracy,F1,FNR
0,Clean baseline,0.94735,0.654187,0.377111
1,Poisoned (5% flip),0.94620,0.627682,0.432770
2,Change (poisoned - clean),-0.00115,-0.026506,0.055660


## Key Question: Operationally More Dangerous Attack

For a live social platform, the **evasion attack is generally more operationally dangerous** in day-to-day moderation.

- **Why it is more dangerous operationally:** Evasion can be executed immediately by ordinary users with no internal access. Attackers can iteratively rephrase toxic content (misspellings, spacing tricks, leetspeak, coded language) until it bypasses filters. At platform scale, even small per-comment effort can produce sustained harm because many users can do it in parallel.
- **Poisoning is high-impact but less common:** Data poisoning can degrade model behavior broadly, but it usually requires privileged access to labeling, data ingestion, or retraining pipelines (or successful compromise of those systems). That is a stronger attacker capability and therefore a less frequent threat model for most platforms.

### Realistic threat model
For most social platforms, the more realistic frequent adversary is an **external user attempting evasion**, not an attacker with training-pipeline control.

### Defense prioritization implication
Defenses should be prioritized in this order:

1. **Inference-time robustness (highest priority):** normalization, adversarial text augmentation during training, ensemble/secondary checks, rate limits, and human review triggers for borderline cases.
2. **Continuous monitoring:** track drift and suspicious bypass patterns; rapidly patch with fresh adversarial examples.
3. **Pipeline security (still essential):** strict data provenance, access control, audit logs, and outlier/anomaly detection in training data to reduce poisoning risk.

In short: **optimize first for resilient real-time moderation against evasion**, while maintaining strong governance and security controls to prevent rarer but potentially catastrophic poisoning events.